In [ ]:
import json
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

In [ ]:
# Configuration

SFT_PATH = Path(
    "data/processed/runs/20260926_183005/data/16_sft_master.jsonl"
)

OUTPUT_PATH = SFT_PATH.parent / "17_sft_explanations.jsonl"

MODEL_ID = "Qwen/Qwen3-1.7B"

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.0

In [ ]:
# Chargement

def read_jsonl(path: Path) -> list[dict]:
    """Charge un JSONL."""
    with open(path, encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def write_jsonl(path: Path, records: list[dict]) -> None:
    """Écrit un JSONL."""
    with open(path, "w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


records = read_jsonl(SFT_PATH)

print(f"SFT : {len(records):,}")

In [ ]:
# Helpers

def language_of(record: dict) -> str:
    """Retourne la langue du record."""
    return str(
        record.get("language")
        or record.get("detected_language")
        or "fr"
    ).lower()


def priority_of(record: dict) -> int:
    """Retourne uniquement la priorité déjà validée."""
    priority = (
        record.get("final_priority")
        or record.get("priority")
        or (record.get("triage_decision") or {}).get("priority")
        or (record.get("triage_engine_decision") or {}).get("priority")
    )

    if priority not in {1, 2, 3}:
        raise ValueError(
            f"Priorité invalide pour {record.get('id')}: {priority}"
        )

    return priority

In [ ]:
# Construction du contexte clinique

CLINICAL_FIELDS = (
    "context",
    "question",
    "answer",
    "patient",
    "medical_knowledge"
)


def clinical_content(record: dict) -> str:
    """Construit le contexte uniquement depuis le record."""
    parts = []

    for field in CLINICAL_FIELDS:
        value = record.get(field)

        if value in (None, "", [], {}):
            continue

        if isinstance(value, (dict, list)):
            value = json.dumps(
                value,
                ensure_ascii=False
            )

        parts.append(
            f"{field.upper()}:\n{value}"
        )

    return "\n\n".join(parts)

In [ ]:
# Prompt
# Génère une explication clinique fondée uniquement sur le record.
# La priorité reste un contexte interne et ne doit jamais apparaître dans la réponse.

def build_prompt(record: dict) -> str:
    """Construit la consigne d'explication clinique."""
    language = language_of(record)
    priority = priority_of(record)
    clinical = clinical_content(record)

    if language == "en":
        instruction = f"""
OBJECTIVE

Write a genuine clinical explanation of the level of severity using
the information available in the record.

The validated severity context is P{priority}. Use it only to guide the
severity of the explanation. Never mention it in the output.

Do not simply list or rephrase the symptoms.

When supported by the available information, explain:
- why a symptom is clinically important;
- why its intensity, duration, or progression is relevant;
- why the combination of several symptoms is significant;
- how a vital sign or clinical result contributes to severity;
- why a described disease or condition may be concerning;
- why an explicitly documented complication is important;
- how several elements present in the record relate to one another.

The explanation must show the clinical relationship between the relevant
elements in the record and the severity of the situation.

STRICT CONSTRAINTS

- The provided priority is definitive and is used only as context.
- Never mention P1, P2, P3, priority, classification, or triage.
- Use only information present in the record.
- Do not add external medical knowledge.
- Do not invent symptoms, diagnoses, results, complications, or mechanisms.
- Do not create causal relationships that are not supported by the
  available information.
- Strictly distinguish observations concerning the patient from general
  medical knowledge contained in the record.
- A general characteristic of a disease must never be presented as
  something observed in the patient.
- Prefer elements that genuinely explain severity rather than summarizing
  the entire record.
- If the record does not support a precise clinical relationship, remain
  descriptive rather than inventing one.
- Do not provide recommendations, referrals, instructions, management
  advice, or treatment.
- Write 1 to 3 natural and informative sentences.
- Vary sentence structure naturally.
- Write exclusively in English.
- Return only the explanation.

CLINICAL INFORMATION

{clinical}
"""

    else:
        instruction = f"""
OBJECTIF

Rédige une véritable explication clinique du niveau de gravité à partir
des informations disponibles dans le record.

Le contexte de gravité validé est P{priority}. Utilise-le uniquement pour
orienter le niveau de gravité de l'explication. Ne le mentionne jamais
dans la réponse.

Ne te contente pas d'énumérer ou de reformuler les symptômes.

Explique, lorsque les informations disponibles le permettent :
- pourquoi un symptôme est important ;
- pourquoi son intensité, sa durée ou son évolution est pertinente ;
- pourquoi l'association de plusieurs symptômes est significative ;
- en quoi une constante ou un résultat clinique contribue à la gravité ;
- en quoi une maladie ou une pathologie décrite peut être préoccupante ;
- en quoi une complication explicitement documentée est importante ;
- comment plusieurs éléments présents dans le record sont liés entre eux.

L'explication doit faire apparaître la relation clinique entre les éléments
pertinents du record et la gravité de la situation.

CONTRAINTES STRICTES

- La priorité fournie est définitive et sert uniquement de contexte.
- Ne mentionne jamais P1, P2, P3, la priorité, le classement ou le triage.
- Utilise uniquement les informations présentes dans le record.
- N'ajoute aucune connaissance médicale externe.
- N'invente aucun symptôme, diagnostic, résultat, complication ou mécanisme.
- Ne crée pas de relation causale qui n'est pas soutenue par les informations
  disponibles.
- Distingue strictement les observations concernant le patient des
  connaissances médicales générales présentes dans le record.
- Une caractéristique générale d'une maladie ne doit jamais être présentée
  comme observée chez le patient.
- Privilégie les éléments qui expliquent réellement la gravité plutôt que
  de résumer tout le dossier.
- Si le record ne permet pas d'établir une relation clinique précise,
  reste descriptif plutôt que d'en inventer une.
- Ne donne aucune recommandation, orientation, conduite à tenir ou traitement.
- Produis 1 à 3 phrases naturelles et informatives.
- Varie naturellement la construction des phrases.
- Écris exclusivement en français.
- Retourne uniquement l'explication.

INFORMATIONS CLINIQUES

{clinical}
"""

    return instruction.strip()

In [ ]:
# Modèle

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)

model.eval()

print(f"Modèle : {MODEL_ID}")
print(f"Device : {model.device}")

In [ ]:
# Génération

def generate_explanation(record: dict) -> str:
    """Génère uniquement l'explication."""
    prompt = build_prompt(record)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

In [ ]:
# Enrichissement

enriched_records = []

for record in tqdm(
    records,
    desc="Explications SFT"
):
    explanation = generate_explanation(record)

    enriched = {
        **record,
        "triage_explanation": {
            "priority": priority_of(record),
            "explanation": explanation,
            "model": MODEL_ID
        }
    }

    enriched_records.append(enriched)

In [ ]:
# Intégrité

if len(enriched_records) != len(records):
    raise RuntimeError(
        "Le nombre de records a changé."
    )

for source, enriched in zip(
    records,
    enriched_records,
    strict=True
):
    if source["id"] != enriched["id"]:
        raise RuntimeError(
            "Un identifiant a changé."
        )

    if priority_of(source) != priority_of(enriched):
        raise RuntimeError(
            f"Priorité modifiée : {source['id']}"
        )

    explanation = (
        enriched.get("triage_explanation") or {}
    ).get("explanation")

    if not explanation:
        raise RuntimeError(
            f"Explication absente : {source['id']}"
        )

print(
    f"✓ {len(enriched_records):,} records validés"
)

print(
    "✓ Priorités inchangées"
)

print(
    "✓ Explications présentes"
)

In [ ]:
# Export

write_jsonl(
    OUTPUT_PATH,
    enriched_records
)

print(
    f"✓ {OUTPUT_PATH.resolve()}"
)